<a href="https://colab.research.google.com/github/arnavon2005/Army_Provost_ML_Project/blob/main/05_Machine_Learning_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Model Development

> **Project:** Army Provost Machine Learning Project  
> **Dataset:** Chicago Crimes Dataset  
> **Notebook:** 05_Machine_Learning_Modeling.ipynb

## Objective

This notebook focuses on developing and evaluating machine learning models for crime analysis.

The primary objective is to build a predictive model that estimates whether a reported crime incident is likely to result in an arrest.

The notebook includes:

- Dataset Loading
- Feature Engineering
- Feature Selection
- Feature Encoding
- Model Training
- Model Evaluation
- Model Comparison
- Model Saving

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Import Required Libraries
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

# Models

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    roc_auc_score,
    RocCurveDisplay
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [3]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_PATH = Path("/content/drive/MyDrive/Army_Provost_ML_Project")

DATA_PATH = PROJECT_PATH / "Datasets"

RAW_DATASET_PATH = DATA_PATH / "Raw"
CLEAN_DATASET_PATH = DATA_PATH / "Cleaned"

FIGURES_PATH = PROJECT_PATH / "Figures"
REPORTS_PATH = PROJECT_PATH / "Reports"
MODELS_PATH = PROJECT_PATH / "Models"
OUTPUTS_PATH = PROJECT_PATH / "Outputs"
LOGS_PATH = PROJECT_PATH / "Logs"

# Create folders if they don't exist
for folder in [
    FIGURES_PATH,
    REPORTS_PATH,
    MODELS_PATH,
    OUTPUTS_PATH,
    LOGS_PATH
]:
    folder.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("Project directories configured successfully.")
print("=" * 70)
print(f"Project : {PROJECT_PATH}")
print(f"Cleaned Data : {CLEAN_DATASET_PATH}")
print(f"Models : {MODELS_PATH}")
print(f"Figures : {FIGURES_PATH}")
print("=" * 70)

Project directories configured successfully.
Project : /content/drive/MyDrive/Army_Provost_ML_Project
Cleaned Data : /content/drive/MyDrive/Army_Provost_ML_Project/Datasets/Cleaned
Models : /content/drive/MyDrive/Army_Provost_ML_Project/Models
Figures : /content/drive/MyDrive/Army_Provost_ML_Project/Figures


In [4]:
# ============================================================
# Load Cleaned Dataset
# ============================================================

print("=" * 70)
print("Loading Cleaned Dataset")
print("=" * 70)

chicago_df = pd.read_csv(
    CLEAN_DATASET_PATH / "chicago_crimes_cleaned.csv",
    low_memory=False
)

print("\nDataset Loaded Successfully!\n")

print(f"Shape : {chicago_df.shape}")

display(chicago_df.head())

Loading Cleaned Dataset

Dataset Loaded Successfully!

Shape : (8602734, 22)


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,14269678,JK342166,2026-07-20 00:00:00,001XX E WACKER DR,0281,CRIMINAL SEXUAL ASSAULT,NON-AGGRAVATED,HOTEL / MOTEL,False,False,...,42.0,32.0,02,1177683.0,1902638.0,2026,2026-07-27 15:43:52,41.888165,-87.622940,"(41.888165132, -87.622937212)"
1,14267487,JK340097,2026-07-20 00:00:00,084XX S MARYLAND AVE,1310,CRIMINAL DAMAGE,TO PROPERTY,RESIDENCE,False,False,...,8.0,44.0,14,1183361.0,1849252.0,2026,2026-07-27 15:43:52,41.741540,-87.603750,"(41.741539131, -87.603750482)"
2,14267485,JK340089,2026-07-20 00:00:00,105XX S WABASH AVE,0486,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,False,True,...,9.0,49.0,08B,1178438.0,1835166.0,2026,2026-07-27 15:43:52,41.703000,-87.622215,"(41.70299844, -87.622214313)"
3,14268664,JK341413,2026-07-20 00:00:00,091XX S LAFAYETTE AVE,0810,THEFT,OVER $500,RESIDENCE,False,True,...,9.0,49.0,06,1177542.0,1844332.0,2026,2026-07-27 15:43:52,41.728172,-87.625220,"(41.728171435, -87.625219193)"
4,14269513,JK342626,2026-07-20 00:00:00,028XX N KIMBALL AVE,4387,OTHER OFFENSE,VIOLATE ORDER OF PROTECTION,APARTMENT,False,False,...,35.0,21.0,26,1153219.0,1919032.0,2026,2026-07-27 15:43:52,41.933674,-87.712340,"(41.933672358, -87.712341606)"


# ============================================================
# Feature Engineering
# ============================================================

## Objective

The objective of this phase is to prepare the dataset for machine learning by
selecting appropriate predictor variables, creating useful time-based features,
and removing columns that are either identifiers, redundant, or could introduce
data leakage.

Proper feature engineering is one of the most important stages of any machine
learning project because the quality of the input features directly affects the
performance and generalization ability of the final model.

## Why is Feature Engineering Important?

Machine learning algorithms cannot automatically determine which columns are
useful and which may negatively affect prediction performance.

During this phase we will:

- Select the target variable.
- Select predictor variables.
- Remove identifier and leakage columns.
- Verify data types.
- Prepare the dataset for encoding.

This step ensures that the model learns meaningful crime patterns instead of
memorizing record-specific information.

## Expected Outcome

By the end of this section we will have a clean feature matrix (X) and target
variable (y) ready for preprocessing and encoding.

## Extracting Time-Based Features

The original dataset contains a `Date` column with both date and time information.
Many machine learning algorithms benefit from explicit temporal features rather than
a single timestamp.

In this step, we convert the `Date` column to a datetime object (if required) and
derive the following features:

- Month
- Day
- Hour

These features will be used as predictor variables in later stages of the project.

In [7]:
# ============================================================
# Extract Time-Based Features
# ============================================================

print("=" * 70)
print("Creating Time-Based Features")
print("=" * 70)

# Convert Date column to datetime (safe even if already converted)
chicago_df["Date"] = pd.to_datetime(chicago_df["Date"])

# Extract temporal features
chicago_df["Month"] = chicago_df["Date"].dt.month
chicago_df["Day"] = chicago_df["Date"].dt.day
chicago_df["Hour"] = chicago_df["Date"].dt.hour

print("\nTime-based features created successfully.")

print("\nNew Columns Added:")
print(["Month", "Day", "Hour"])

print("\nPreview:")
print(chicago_df[["Date", "Month", "Day", "Hour"]].head())

Creating Time-Based Features

Time-based features created successfully.

New Columns Added:
['Month', 'Day', 'Hour']

Preview:
        Date  Month  Day  Hour
0 2026-07-20      7   20     0
1 2026-07-20      7   20     0
2 2026-07-20      7   20     0
3 2026-07-20      7   20     0
4 2026-07-20      7   20     0


In [8]:
# ============================================================
# Feature Engineering
# ============================================================

print("=" * 70)
print("Feature Engineering")
print("=" * 70)

# -----------------------------
# Target Variable
# -----------------------------
TARGET_COLUMN = "Arrest"

# -----------------------------
# Selected Predictor Columns
# -----------------------------
FEATURE_COLUMNS = [
    "Primary Type",
    "Description",
    "Location Description",
    "Domestic",
    "Year",
    "Month",
    "Day",
    "Hour",
    "District",
    "Beat",
    "Ward",
    "Community Area"
]

# -----------------------------
# Create Feature Matrix & Target
# -----------------------------
X = chicago_df[FEATURE_COLUMNS].copy()
y = chicago_df[TARGET_COLUMN].copy()

print(f"\nTarget Variable : {TARGET_COLUMN}")
print(f"Number of Features : {len(FEATURE_COLUMNS)}")

print("\nSelected Features:")
for i, feature in enumerate(FEATURE_COLUMNS, start=1):
    print(f"{i:>2}. {feature}")

print("\nFeature Matrix Shape :", X.shape)
print("Target Shape         :", y.shape)

Feature Engineering

Target Variable : Arrest
Number of Features : 12

Selected Features:
 1. Primary Type
 2. Description
 3. Location Description
 4. Domestic
 5. Year
 6. Month
 7. Day
 8. Hour
 9. District
10. Beat
11. Ward
12. Community Area

Feature Matrix Shape : (8602734, 12)
Target Shape         : (8602734,)


## Feature Validation

Before applying encoding and training machine learning models, it is important to
validate the selected predictor variables.

During this step we will:

- Verify the data types of all selected features.
- Check for missing values.
- Ensure the target variable contains valid binary classes.

This validation helps prevent errors during preprocessing and model training while
maintaining a reliable and reproducible machine learning pipeline.

In [9]:
# ============================================================
# Feature Validation
# ============================================================

print("=" * 70)
print("Feature Validation")
print("=" * 70)

# Display feature data types
print("\nFeature Data Types:\n")
print(X.dtypes)

# Missing values
missing_summary = (
    X.isnull()
     .sum()
     .to_frame(name="Missing Values")
)

missing_summary["Percentage"] = (
    missing_summary["Missing Values"] / len(X) * 100
).round(2)

print("\n")
print("=" * 70)
print("Missing Value Summary")
print("=" * 70)

print(missing_summary)

print("\n")
print("=" * 70)
print("Target Distribution")
print("=" * 70)

print(y.value_counts())

print("\n")
print("Target Percentage:")

print(
    (y.value_counts(normalize=True) * 100)
    .round(2)
)

Feature Validation

Feature Data Types:

Primary Type             object
Description              object
Location Description     object
Domestic                   bool
Year                      int64
Month                     int32
Day                       int32
Hour                      int32
District                float64
Beat                      int64
Ward                    float64
Community Area          float64
dtype: object


Missing Value Summary
                      Missing Values  Percentage
Primary Type                       0        0.00
Description                        0        0.00
Location Description               0        0.00
Domestic                           0        0.00
Year                               0        0.00
Month                              0        0.00
Day                                0        0.00
Hour                               0        0.00
District                          47        0.00
Beat                               0        0.0

# ============================================================
# Missing Value Treatment
# ============================================================

## Objective

Before applying categorical encoding and training machine learning models, all
selected predictor variables should be free from missing values.

Machine learning algorithms generally do not handle missing values consistently,
and preprocessing techniques such as one-hot encoding require complete data.

In this step, missing values are imputed using the most frequent (mode) value
for each feature. This approach preserves all observations while maintaining
the categorical nature of geographical variables.

## Imputation Strategy

| Feature | Strategy |
|----------|----------|
| District | Mode |
| Ward | Mode |
| Community Area | Mode |

## Expected Outcome

After this step, the feature matrix will contain no missing values and will be
ready for categorical encoding.

In [10]:
# ============================================================
# Missing Value Treatment
# ============================================================

print("=" * 70)
print("Missing Value Treatment")
print("=" * 70)

# Store missing values before imputation
missing_before = X.isnull().sum()

print("\nMissing Values Before Imputation:\n")
print(missing_before[missing_before > 0])

# ------------------------------------------------------------
# Mode Imputation
# ------------------------------------------------------------

imputation_columns = [
    "District",
    "Ward",
    "Community Area"
]

for column in imputation_columns:
    mode_value = X[column].mode()[0]
    X[column] = X[column].fillna(mode_value)

# ------------------------------------------------------------
# Verify Imputation
# ------------------------------------------------------------

missing_after = X.isnull().sum()

print("\n")
print("=" * 70)
print("Missing Values After Imputation")
print("=" * 70)

print(missing_after[missing_after > 0])

if missing_after.sum() == 0:
    print("\n✅ All missing values have been successfully handled.")
else:
    print("\n⚠️ Some missing values are still present.")

print("\nFinal Dataset Shape:", X.shape)

Missing Value Treatment

Missing Values Before Imputation:

District              47
Ward              614813
Community Area    613724
dtype: int64


Missing Values After Imputation
Series([], dtype: int64)

✅ All missing values have been successfully handled.

Final Dataset Shape: (8602734, 12)


# ============================================================
# Categorical Feature Analysis
# ============================================================

## Objective

Before applying categorical encoding, it is important to understand the
cardinality of each categorical feature.

The number of unique values directly influences the choice of encoding
technique, memory consumption, and model training time.

This analysis will help determine the most appropriate encoding strategy for
each categorical variable.

In [11]:
# ============================================================
# Categorical Feature Analysis
# ============================================================

print("=" * 70)
print("Categorical Feature Analysis")
print("=" * 70)

categorical_columns = [
    "Primary Type",
    "Description",
    "Location Description"
]

summary = []

for column in categorical_columns:
    unique_count = X[column].nunique()

    summary.append({
        "Feature": column,
        "Unique Values": unique_count
    })

summary_df = pd.DataFrame(summary)

print(summary_df)

print("\n")

for column in categorical_columns:
    print("-" * 70)
    print(column)
    print("-" * 70)
    print(X[column].value_counts().head(10))
    print()

Categorical Feature Analysis
                Feature  Unique Values
0          Primary Type             34
1           Description            570
2  Location Description            219


----------------------------------------------------------------------
Primary Type
----------------------------------------------------------------------
Primary Type
THEFT                  1827377
BATTERY                1567212
CRIMINAL DAMAGE         977357
NARCOTICS               768704
ASSAULT                 580094
OTHER OFFENSE           537540
BURGLARY                455436
MOTOR VEHICLE THEFT     444725
DECEPTIVE PRACTICE      400071
ROBBERY                 318103
Name: count, dtype: int64

----------------------------------------------------------------------
Description
----------------------------------------------------------------------
Description
SIMPLE                          1012886
$500 AND UNDER                   687475
DOMESTIC BATTERY SIMPLE          672476
TO VEHICLE            

# ============================================================
# Preprocessing Pipeline
# ============================================================

## Objective

Machine learning models require numerical input data. Since the dataset contains
both categorical and numerical features, an appropriate preprocessing pipeline
must be constructed.

In this section, we:

- Remove the high-cardinality `Description` feature.
- Separate categorical and numerical features.
- Apply One-Hot Encoding to categorical variables.
- Pass numerical variables through without modification.
- Build a reusable preprocessing pipeline using Scikit-learn's
  `ColumnTransformer`.

Using a preprocessing pipeline improves reproducibility, prevents data leakage,
and ensures that the same transformations are applied during both training and
future predictions.

## Expected Outcome

A reusable preprocessing pipeline ready for train-test splitting and model
training.

In [12]:
# ============================================================
# Preprocessing Pipeline
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

print("=" * 70)
print("Building Preprocessing Pipeline")
print("=" * 70)

# ------------------------------------------------------------
# Remove High-Cardinality Feature
# ------------------------------------------------------------

X = X.drop(columns=["Description"])

print("\nRemoved Feature:")
print("Description")

# ------------------------------------------------------------
# Define Feature Groups
# ------------------------------------------------------------

categorical_features = [
    "Primary Type",
    "Location Description",
    "Domestic"
]

numerical_features = [
    "Year",
    "Month",
    "Day",
    "Hour",
    "District",
    "Beat",
    "Ward",
    "Community Area"
]

print("\nCategorical Features:")
for feature in categorical_features:
    print(f"• {feature}")

print("\nNumerical Features:")
for feature in numerical_features:
    print(f"• {feature}")

# ------------------------------------------------------------
# Build ColumnTransformer
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("\n✅ Preprocessing pipeline created successfully.")

print("\nTotal Features Remaining:", X.shape[1])

Building Preprocessing Pipeline

Removed Feature:
Description

Categorical Features:
• Primary Type
• Location Description
• Domestic

Numerical Features:
• Year
• Month
• Day
• Hour
• District
• Beat
• Ward
• Community Area

✅ Preprocessing pipeline created successfully.

Total Features Remaining: 11
